[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/39_ppo_loss.ipynb)

# 🔴 Hard: PPO Clipped Loss

实现 **PPO (Proximal Policy Optimization)** 的 **clipped surrogate loss**。这是当前最常用的基于策略梯度的强化学习目标之一，也经常用于对语言模型进行 RL 微调。

给定：
- `new_logps`: 当前策略 $\pi_\theta$ 产生的 log 概率，形状为 $(B,)$
- `old_logps`: 旧策略 $\pi_{\theta_\text{old}}$ 产生的 log 概率，形状为 $(B,)$
- `advantages`: 已经估计好的优势函数 $A_i$，形状为 $(B,)$

首先定义比例：

$$ r_i = \exp(\log \pi_\theta(y_i) - \log \pi_{\theta_\text{old}}(y_i)) = \exp(\text{new\_logps}_i - \text{old\_logps}_i). $$

然后定义两个目标：

- 未裁剪目标：$L^{\text{unclipped}}_i = r_i A_i$
- 裁剪后的目标：
  $$L^{\text{clipped}}_i = \operatorname{clip}(r_i, 1-\epsilon, 1+\epsilon) A_i$$

PPO clipped loss 取两者中 **更保守** 的那个（对正优势来说更小，对负优势来说更大），再取负号并对 batch 取平均：

$$
\mathcal{L}_\text{PPO} = -\mathbb{E}_i\big[\min\big(L^{\text{unclipped}}_i, L^{\text{clipped}}_i\big)\big].
$$

实现时要注意：
- 把 `old_logps` 和 `advantages` 当作常数，不让梯度回传；
- 只让梯度通过 `new_logps` 反向传播。

### Signature
```python
from torch import Tensor

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    """PPO clipped surrogate loss over a batch.

    new_logps: (B,) current policy log-probs
    old_logps: (B,) old policy log-probs (treated as constant)
    advantages: (B,) advantage estimates (treated as constant)
    returns: scalar loss (Tensor)
    """
```


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    pass  # -mean(min(r * adv, clamp(r, 1-clip, 1+clip) * adv)) with gradients only through new_logps


In [ ]:
# 🧪 Debug
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('ppo_loss')
